# Data Understanding file 2

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../..')))

from src.utils import data_understanding as du

data_frame = pd.read_csv("../../data/raw/laboratory_events_codes_2.csv")

data_frame

In [ ]:
# Shows how many rows and columns the dataset has
print("Dataset shape (rows, columns):", data_frame.shape)


In [ ]:
# Uses the get_cols() function from data_understanding.py to list all columns
print("List of column names:\n")
print(du.get_cols(data_frame))

In [ ]:
print("Data types of each column:\n")
print(du.get_types(data_frame))


TOMMASO QUI SOTTO HO AGGIUNTO LA STAMPA DELLE RIGHE DUPLICATE, da aggiungere nel main alla function del numero dei duplicati

In [ ]:
num_dups = du.get_num_dups(data_frame)
print(f"Number of duplicated rows: {num_dups}")

# Shows which rows are duplicated (if any)
if num_dups > 0:
    print("\nList of duplicated rows:\n")
    print(data_frame[data_frame.duplicated(keep=False)])
else:
    print("\nNo duplicated rows found.")


In [ ]:
print("Columns containing at least one NaN value:\n")
print(du.get_cols_with_none(data_frame))

In [ ]:
def get_none_percentage_per_cols(data_frame):
    res = []
    cols = du.get_cols(data_frame)
    rows = data_frame.shape[0]
    for col in cols:
        not_none = (data_frame[col].count())
        none_percentage = ((rows - not_none)/rows)*100
        res.append((col, none_percentage))
    return res
# Get the percentage of None values for each columns
none_percentage_per_cols = []

print("PERCENTAGE OF NONE PER COLUMN")
for col, percentage in sorted(get_none_percentage_per_cols(data_frame), key=lambda tup: tup[1], reverse=True):
    print(f"{col}: {percentage:.2f}%")

In [ ]:
# Count of None (NaN) values per column
print("NUMBER OF MISSING (none) VALUES PER COLUMN:\n")
none_counts = data_frame.isna().sum().sort_values(ascending=False)
print(none_counts)

none_counts_log = np.log10(none_counts.replace(0, np.nan))

none_df = none_counts_log.reset_index()
none_df.columns = ["column", "log10_count"]

fig, ax = plt.subplots(figsize=(10,8), dpi=300)
sns.barplot(data=none_df, x="column", y="log10_count", color="#99c2ff", edgecolor="black")

plt.title("Number of missing (None) values per column (log10 scale)")
plt.xlabel("Column name")
plt.ylabel("Log10 of count of None values")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
# Add value labels on top of bars
for i, v in enumerate(none_df["log10_count"]):
    if not np.isnan(v):  # evita etichette per colonne senza None
        ax.text(i, v + max(none_df["log10_count"]) * 0.01, 
                f"{v:.2f}", ha="center", va="bottom", fontsize=9)
plt.show()


In [ ]:
numeric_cols = du.get_numeric_cols(data_frame)

print("Numeric columns selected for analysis:\n")
print(numeric_cols)

print("\nDescriptive statistics for numeric columns:\n")
print(du.get_basic_stats(data_frame))


In [ ]:
counts = (
    data_frame.groupby(["fluid", "qc_flag"])
    .size()
    .reset_index(name="count")
)

counts = counts[counts["count"] > 0]
fluid_order = (
    counts.groupby("fluid")["count"]
    .sum()
    .sort_values(ascending=False)
    .index.tolist()
)

counts["log_count"] = np.log10(counts["count"])

plt.figure(figsize=(10,8), dpi=300)
ax = sns.barplot(
    data=counts,
    x="fluid",
    y="log_count",
    hue="qc_flag",
    order=fluid_order,
    palette="Set2",
    edgecolor="black",
)

plt.title('QC flag distribution (log scaled 10)', fontsize=14)
plt.xlabel('Fluid type')
plt.ylabel('log10(Count)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

plt.show()

In [ ]:
required_cols = {"hadm_id", "flag"}

if required_cols.issubset(data_frame.columns):
    # filter rows with flag = "abnormal"
    df_abnormal = (
        data_frame[data_frame["flag"] == "abnormal"]
        .groupby("hadm_id")
        .size()
        .sort_values(ascending=False)
    )

    print("Number of abnormal lab values per hadm_id:\n")
    print(df_abnormal)
else:
    print("Columns required for abnormal value analysis are missing.")


In [ ]:
bins = [0, 1, 10, 500, 1000, 1500, df_abnormal.max()]
labels = ["=1", "2–10", "11–500", "501–1000", "1001–1500", ">1500"]

categories = pd.cut(df_abnormal.values, bins=bins, labels=labels, include_lowest=True)

# how many hamd_id has tot flag="abnormal"
distribution = pd.Series(categories).value_counts(sort=False)

print("Distribuzione per fasce del numero di valori anomali per ammissione:\n")
print(distribution)


In [ ]:
# Define the bins and labels for count ranges
bins = [0, 1, 10, 500, 1000, 1500, df_abnormal.max()]
labels = ["=1", "2–10", "11–500", "501–1000", "1001–1500", ">1500"]

# Reset index to get hadm_id as a column and count as another column
abnormal_counts = df_abnormal.reset_index()
abnormal_counts.columns = ["hadm_id", "count"]

# Bin the count values into specified ranges
abnormal_counts["count_bin"] = pd.cut(
    abnormal_counts["count"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Group by bin and count unique hadm_id in each bin
distribution_counts = (
    abnormal_counts.groupby("count_bin", observed=False)["hadm_id"]
    .nunique()
    .reindex(labels, fill_value=0)
    .reset_index(name="admissions")
)

# Plot configuration with seaborn and matplotlib
sns.set_theme(style="whitegrid", rc={"axes.facecolor": "#F9FAFB"})
fig, ax = plt.subplots(figsize=(10, 8), dpi=300)

bar = sns.barplot(
    data=distribution_counts,
    x="count_bin",
    y="admissions",
    color="#99c2ff",
    edgecolor="#1F2933"
)

# Title and axis labels
ax.set_title("Abnormal distribution for hadmission", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of abnormal values per hadm_id")
ax.set_ylabel("Number of admissions")

# Grid and ticks styling
ax.grid(axis="y", color="#D1D5DB", linestyle="--", linewidth=0.8)
ax.tick_params(axis="x", labelrotation=45)
plt.setp(ax.get_xticklabels(), ha="right")

sns.despine(ax=ax, top=True, right=True)

# Annotate bars with admission counts
for patch, (_, row) in zip(bar.patches, distribution_counts.iterrows()):
    ax.text(
        patch.get_x() + patch.get_width() / 2,
        patch.get_height() + max(distribution_counts["admissions"]) * 0.02,
        f"{row.admissions}",
        ha="center",
        va="bottom",
        fontsize=9,
        color="#111827"
    )

plt.tight_layout()
plt.show()


In [ ]:
# Checks consistency between ref_range_lower and ref_range_upper
# Filters rows with inverted or equal reference ranges
inconsistent = data_frame[data_frame["ref_range_lower"] > data_frame["ref_range_upper"]]
equal_bounds = data_frame[data_frame["ref_range_lower"] == data_frame["ref_range_upper"]]

print(f"Inconsistent ranges: {len(inconsistent)}")
print(f"Equal lower/upper bounds: {len(equal_bounds)}")

In [ ]:
# Checks how many lab results fall outside their reference range
# Filters only rows with valid numeric values and reference limits
valid = data_frame.dropna(subset=["valuenum", "ref_range_lower", "ref_range_upper"])

# Selects those with valuenum below lower limit or above upper limit
outside = valid[
    (valid["valuenum"] < valid["ref_range_lower"]) |
    (valid["valuenum"] > valid["ref_range_upper"])
]

ratio = len(outside) / len(valid)
print(f"Out-of-range values: {len(outside)} / {len(valid)} ({ratio:.2%})")


In [ ]:
correlation_matrix = du.get_correlation(data_frame)

if correlation_matrix is not None:
    print("Kendall correlation between numeric columns:\n")
    print(correlation_matrix)
else:
    print("Not enough numeric columns to calculate correlation.")


In [ ]:
print("General dataset information:")
du.get_info(data_frame)

In [ ]:
# Shows how many different units of measurement are present (mg/dL, mmol/L, etc.)
print("Distribution of measurement units (valueuom):\n")
print(data_frame["valueuom"].value_counts(dropna=False))


In [ ]:
# Displays how many tests are flagged as abnormal
print("Abnormality flag counts (flag column):\n")
print(data_frame["flag"].value_counts(dropna=False))


In [ ]:
# Shows the number of tests performed for each fluid type (e.g., Blood, Urine, etc.)
print("Number of tests per fluid type:\n")
print(data_frame["fluid"].value_counts(dropna=False))

In [ ]:
fluid_counts = (
    data_frame.groupby("fluid", dropna=False)["hadm_id"]
    .nunique()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
fluid_counts["log10_count"] = np.log10(fluid_counts["count"].replace(0, np.nan))

plt.figure(figsize=(10, 8), dpi=300)
ax = sns.barplot(
    data=fluid_counts,
    x="fluid",
    y="log10_count",
    color="#99c2ff",
    edgecolor="black",
)

ax.set_title("Number of unique admissions per fluid type (log10 scale)", fontsize=14, fontweight="bold")
ax.set_xlabel("Fluid type")
ax.set_ylabel("Number of unique hadm_id (log10 scale)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

for patch, (_, row) in zip(ax.patches, fluid_counts.iterrows()):
    if not np.isnan(row["log10_count"]):
        ax.text(
            patch.get_x() + patch.get_width() / 2,
            row["log10_count"] + 0.02,
            f"{row['count']:,}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
plt.tight_layout()
plt.show()

In [ ]:
# frequencies list of laboratory test names in the dataset
print("frequencies list of laboratory test names:\n")
print(data_frame["label"].value_counts())


In [ ]:
# Shows basic descriptive statistics for the numeric test results (valuenum) grouped by lab test label
if "valuenum" in data_frame.columns and "label" in data_frame.columns:
    print("Descriptive statistics for 'valuenum' grouped by lab test label:\n")
    print(data_frame.groupby("label")["valuenum"].describe())
else:
    print("Columns 'valuenum' or 'label' not found in the dataset.")

In [ ]:
#analysis of valuenum for each label 
value=data_frame.groupby("label")["valuenum"].describe()
value=value[['min','max']]
value

In [ ]:
# Displays how many lab results have each qc_flag status (OK, WARN, FAIL)
print("Quality control flag distribution:\n")
print(data_frame["qc_flag"].value_counts(dropna=False))


In [ ]:
qc_counts = data_frame["qc_flag"].value_counts(dropna=False)

qc_df = qc_counts.reset_index()
qc_df.columns = ["qc_flag", "count"]

plt.figure(figsize=(7,5))
sns.barplot(data=qc_df, x="qc_flag", y="count", color="#99c2ff", edgecolor="black")

plt.title("Distribuzione dei valori di Quality Control (qc_flag)")
plt.xlabel("QC Flag")
plt.ylabel("Numero di risultati")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Shows how many missing entries are in valuenum, ref_range_lower, and ref_range_upper
key_cols = ["valuenum", "ref_range_lower", "ref_range_upper"]
print("Missing values in key numeric columns:\n")
print(data_frame[key_cols].isnull().sum())


# PLOT 

In [ ]:
#per avere il numero per range preciso se puo interessarci (resa visiva due celle sotto)

bins = [0, 50, 100, 200, 300, 500, 1000, 5000, data_frame["valuenum"].max()]
labels = ["0-50", "50-100", "100-200", "200-300", "300-500", "500-1000", "1000-5000", "5000+"]

data_frame["valuenum_range"] = pd.cut(data_frame["valuenum"], bins=bins, labels=labels, include_lowest=True)
counts = data_frame["valuenum_range"].value_counts().sort_index()

print("Conteggio dei valori per intervallo:")
print(counts)


analisi personale (e banale) di Sara: La distribuzione dei risultati di laboratorio è fortemente sbilanciata: oltre il 70 % dei valori numerici si concentra sotto 50 e quasi il 96 % sotto 300.
Le code alte (oltre 1000, per un massimo di 900000) rappresentano meno dello 0.5 % dei casi e possono essere considerate outlier o dovute a unità di misura differenti.

In [ ]:
plot_df = (
    data_frame.dropna(subset=["fluid", "valuenum"])
    .loc[lambda df: df["valuenum"] > 0]
)

fluid_order = (
    plot_df.groupby("fluid")["valuenum"]
    .median()
    .sort_values(ascending=False)
    .index
)

plt.figure(figsize=(12, 8), dpi=300)
ax = sns.boxplot(
    data=plot_df,
    x="fluid",
    y=plot_df["valuenum"],
    order=fluid_order,
    palette="pastel",
    showfliers=True,
)

ax.set_title("Log10 distribution of lab values by fluid type", fontsize=14, fontweight="bold")
ax.set_xlabel("Fluid type")
ax.set_ylabel("log10(valuenum)")
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
bin_edges = [0, 1, 10, 500, 1000, 1500, np.inf]
bin_labels = ["=1", "2–10", "11–500", "501–1000", "1001–1500", ">1500"]

abnormal_counts = (
    df_abnormal.reset_index()
    .rename(columns={0: "count"})
    .drop_duplicates(subset=["hadm_id"])
)
abnormal_counts["count_bin"] = pd.cut(
    abnormal_counts["count"],
    bins=bin_edges,
    labels=bin_labels,
    include_lowest=True,
    right=True,
)

distribution_counts = (
    abnormal_counts.groupby("count_bin", observed=False)["hadm_id"]
    .nunique()
    .reindex(bin_labels, fill_value=0)
    .reset_index(name="admissions")
)

print("Distribuzione per fasce del numero di valori anomali per ammissione:")
print(distribution_counts.set_index("count_bin")["admissions"])

plt.figure(figsize=(10, 6), dpi=300)
bar = sns.barplot(
    data=distribution_counts,
    x="count_bin",
    y="admissions",
    color="#99c2ff",
    edgecolor="#1F2933",
)

plt.title("Distribuzione per fasce del numero di valori anomali per ammissione", fontsize=14, fontweight="bold")
plt.xlabel("Numero di valori anomali per hadm_id")
plt.ylabel("Numero di ammissioni")
plt.xticks(rotation=0)

max_height = distribution_counts["admissions"].max()
for patch, (_, row) in zip(bar.patches, distribution_counts.iterrows()):
    height = patch.get_height()
    ax_y = height + (0.02 * max_height if height > 0 else max_height * 0.02)
    bar.annotate(
        f"{row.admissions}",
        (patch.get_x() + patch.get_width() / 2, ax_y),
        ha="center",
        va="bottom",
        fontsize=9,
        color="#111827",
    )

plt.tight_layout()
plt.show()